In [1]:
# Cell 1: Imports and setup
import os, random
from pathlib import Path
from collections import defaultdict
import pandas as pd
import yaml

# Load config
ROOT = Path.cwd().parent if 'notebooks' in Path.cwd().parts else Path.cwd()
CONFIG_PATH = ROOT / 'config' / 'config.yaml'
cfg = yaml.safe_load(open(CONFIG_PATH, 'r'))

# Paths
RAW_DATA_DIR = ROOT / cfg['config']['raw_dir']
SPLITS_DIR = ROOT / 'data' / 'splits'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Config params
classes = cfg['config']['classes']
train_ratio = cfg['config']['train_split']
val_ratio = cfg['config']['val_split']
test_ratio = cfg['config']['test_split']

print(f"ROOT: {ROOT}")
print(f"Classes: {classes}")
print(f"Splits: {train_ratio}/{val_ratio}/{test_ratio}")


ROOT: /Users/harryirving/Development/projects/ai-ml/BikeAIv3
Classes: ['angle_grinder', 'background_noise', 'power_tools']
Splits: 0.7/0.15/0.15


In [2]:
# Cell 2: Collect files per class
files_by_class = defaultdict(list)

for cls in classes:
    cls_dir = RAW_DATA_DIR / cls
    if cls_dir.exists():
        audio_files = list(cls_dir.glob('*.wav')) + list(cls_dir.glob('*.mp3')) + list(cls_dir.glob('*.flac'))
        files_by_class[cls] = audio_files
        print(f"{cls}: {len(audio_files)} files")
    else:
        print(f"⚠️ WARNING: {cls_dir} not found!")

total_files = sum(len(files) for files in files_by_class.values())
print(f"\nTotal files: {total_files}")


angle_grinder: 30 files
background_noise: 1347 files
power_tools: 24 files

Total files: 1401


In [3]:
# Cell 3: File-level split (NO DATA LEAKAGE)
train_files = []
val_files = []
test_files = []

random.seed(42)  # Reproducibility

for cls, files in files_by_class.items():
    random.shuffle(files)
    
    n = len(files)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    # Split files (not segments)
    train_files.extend([(f, cls) for f in files[:train_end]])
    val_files.extend([(f, cls) for f in files[train_end:val_end]])
    test_files.extend([(f, cls) for f in files[val_end:]])

print(f"File-level split:")
print(f"Train: {len(train_files)} files")
print(f"Val: {len(val_files)} files")
print(f"Test: {len(test_files)} files")


File-level split:
Train: 979 files
Val: 210 files
Test: 212 files


In [4]:
# Cell 4: Save splits to manifests
def write_manifest(file_list, output_path):
    with open(output_path, 'w') as f:
        for filepath, label in file_list:
            rel_path = filepath.relative_to(RAW_DATA_DIR)
            f.write(f"{rel_path},{label}\n")

write_manifest(train_files, SPLITS_DIR / "train.txt")
write_manifest(val_files, SPLITS_DIR / "val.txt")
write_manifest(test_files, SPLITS_DIR / "test.txt")

print(f"✓ Splits saved to {SPLITS_DIR}")


✓ Splits saved to /Users/harryirving/Development/projects/ai-ml/BikeAIv3/data/splits


In [5]:
# Cell 5: Verify no data leakage
train_stems = set([Path(f).stem for f, _ in train_files])
val_stems = set([Path(f).stem for f, _ in val_files])
test_stems = set([Path(f).stem for f, _ in test_files])

overlap_train_test = train_stems & test_stems
overlap_train_val = train_stems & val_stems
overlap_val_test = val_stems & test_stems

print(f"Train-Test overlap: {len(overlap_train_test)} (should be 0)")
print(f"Train-Val overlap: {len(overlap_train_val)} (should be 0)")
print(f"Val-Test overlap: {len(overlap_val_test)} (should be 0)")

if len(overlap_train_test) == 0 and len(overlap_train_val) == 0 and len(overlap_val_test) == 0:
    print("✅ No data leakage detected!")
else:
    print("⚠️ WARNING: Data leakage detected!")
    print(f"Overlapping files: {overlap_train_test | overlap_train_val | overlap_val_test}")


Train-Test overlap: 0 (should be 0)
Train-Val overlap: 1 (should be 0)
Val-Test overlap: 0 (should be 0)
⚠️ WARNING: Data leakage detected!
Overlapping files: {'angle-grinder-313298'}


In [6]:
# Cell 6: Show class distribution per split
def show_distribution(file_list, split_name):
    labels = [label for _, label in file_list]
    dist = pd.Series(labels).value_counts()
    print(f"\n{split_name} distribution:")
    print(dist)
    return dist

train_dist = show_distribution(train_files, "Train")
val_dist = show_distribution(val_files, "Val")
test_dist = show_distribution(test_files, "Test")



Train distribution:
background_noise    942
angle_grinder        21
power_tools          16
Name: count, dtype: int64

Val distribution:
background_noise    202
angle_grinder         4
power_tools           4
Name: count, dtype: int64

Test distribution:
background_noise    203
angle_grinder         5
power_tools           4
Name: count, dtype: int64
